# Imports

In [1]:
from WS_Mdl.core import *

In [2]:
import datetime as DT
import pandas as pd

# Prep

In [3]:
MdlN = 'NBr70'
MdlN_B = 'NBr68'

In [4]:
M = Mdl_N(MdlN)
MB = Mdl_N(MdlN_B)

# Load + process DF

In [5]:
Pa_SFR_Out = MB.Pa.Sim_In / f'{MB.MdlN}.SFR6.obs.output.csv'
DF = pd.read_csv(Pa_SFR_Out)

In [6]:
DF['date'] = pd.to_datetime(DF['time'], unit='d', origin=pd.Timestamp(MB.INI.SDATE)) - DT.timedelta(days=1) # Convert time to date

In [7]:
DF['month'] = DF['date'].dt.month

In [8]:
DF = DF[['date', 'month'] + [i for i in DF.columns if 'STG' in i]] # Only keep stage columns

In [9]:
DF.loc['AVG_Stg_summer'] = DF.loc[(DF['month']>3) & (DF['month']<10)].copy().mean(axis='index')
DF.loc['AVG_Stg_winter'] = DF.loc[(DF['month']>=10) | (DF['month']<=3)].copy().mean(axis='index')
DF.loc['AVG_Stg_winter_m_summer'] = DF.loc['AVG_Stg_winter'] - DF.loc['AVG_Stg_summer']

In [10]:
DF.drop([i for i in DF.index if i not in ['AVG_Stg_summer', 'AVG_Stg_winter', 'AVG_Stg_winter_m_summer']], inplace=True)

In [11]:
DF = DF.drop(['date', 'month'], axis='columns').T

In [12]:
DF[['L', 'R', 'C']] = DF.index.to_series().str.extract(r'L(\d+)_R(\d+)_C(\d+)').astype(int)

In [13]:
DF = DF.ws.Calc_XY(MB.Xmin, MB.Ymax, MB.cellsize)

# Conert to DA

In [14]:
DA = DF.set_index(['y', 'x'])[['AVG_Stg_summer', 'AVG_Stg_winter', 'AVG_Stg_winter_m_summer']].to_xarray()

# Save stage .IDFs

In [ ]:
import imod
import rioxarrray

In [50]:
for Par in ['AVG_Stg_summer', 'AVG_Stg_winter', 'AVG_Stg_winter_m_summer']:
    DA_ = DA[Par].rio.write_crs('EPSG:28992') 
    DA_.rio.to_raster(MB.Pa.PoP_Out_MdlN / f'SFR/Stg/SFR_{Par}_{MB.MdlN}.TIF')

# Save SFR+RIV stage .IDFs
We need to do this cause the SFR only has values inside the catchment.

## Load RIV

In [27]:
from WS_Mdl.xr.spatial import clip_Mdl_area

In [35]:
import xarray as xra

In [31]:
RIV_Stg_summer = clip_Mdl_area(imod.idf.open(M.Pa.Mdl / r"In\RIV\NBr49\RIV_Stg_main_summer_NBr49.IDF"), M.MdlN)
RIV_Stg_winter = clip_Mdl_area(imod.idf.open(M.Pa.Mdl / r"In\RIV\NBr49\RIV_Stg_main_winter_NBr49.IDF"), M.MdlN)

🟡 - Reversed y-axis of DataArray to match model area orientation.
🟡 - Reversed y-axis of DataArray to match model area orientation.


# Align Coordinates, Join, Save

In [59]:
RIV_Stg_winter, SFR_Stg_winter = xra.align(RIV_Stg_winter, DA['AVG_Stg_winter'], join='outer')
RIV_Stg_summer, SFR_Stg_summer = xra.align(RIV_Stg_summer, DA['AVG_Stg_summer'], join='outer')

In [60]:
imod.idf.save(M.Pa.In / f'RIV/{MdlN}/RIV_Stg_main_winter_{M.MdlN}.IDF', xra.where(SFR_Stg_winter.notnull(), SFR_Stg_winter, RIV_Stg_winter))
imod.idf.save(M.Pa.In / f'RIV/{MdlN}/RIV_Stg_main_summer_{M.MdlN}.IDF', xra.where(SFR_Stg_summer.notnull(), SFR_Stg_summer, RIV_Stg_summer))